# Instructor Effectiveness Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('instructor_effectiveness_dataset_2000_rows - instructor_effectiveness_dataset_2000_rows.csv.csv')
df.head()

Let's check the data info and see if there are any missing values.

In [ ]:
df.info()
df.isnull().sum()

Just plotting a few distributions to see what we are working with.

In [ ]:
df['completion_rate'].hist(bins=20)
plt.title('Completion Rate')
plt.show()

df['avg_feedback_score'].hist(bins=20)
plt.title('Feedback Score')
plt.show()

### 2. Defining Instructor Effectiveness
I need to come up with a score. I'll combine the learner outcomes, engagement, and the feedback.
I'll use MinMaxScaler so all values are on a similar scale before adding them up.

Outcomes: completion_rate, avg_score_improvement
Engagement: avg_watch_time, assignment_submission_rate
Feedback: avg_feedback_score

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cols = ['avg_score_improvement', 'avg_feedback_score']
df_scaled = df.copy()
df_scaled[cols] = scaler.fit_transform(df[cols])

# calculating a weighted sum for effectiveness
df['score'] = (
    0.2 * df['completion_rate'] + 
    0.2 * df_scaled['avg_score_improvement'] +
    0.15 * df['avg_watch_time'] +
    0.15 * df['assignment_submission_rate'] +
    0.3 * df_scaled['avg_feedback_score']
)

# finding the cutoffs for low, medium, high
cutoffs = df['score'].quantile([0.33, 0.67]).values

def get_tier(s):
    if s <= cutoffs[0]: return 'Low'
    elif s <= cutoffs[1]: return 'Medium'
    else: return 'High'
    
df['tier'] = df['score'].apply(get_tier)
df['tier'].value_counts()

### 3. Aggregating to instructor level
Since instructors teach multiple batches, I will just take the average of all their batches. That seems like the most straightforward way to evaluate them overall.

In [ ]:
instructor_df = df.groupby('instructor_id').mean(numeric_only=True).reset_index()

# recalculate the tier for the instructor based on their average score
agg_cutoffs = instructor_df['score'].quantile([0.33, 0.67]).values

def get_agg_tier(s):
    if s <= agg_cutoffs[0]: return 'Low'
    elif s <= agg_cutoffs[1]: return 'Medium'
    else: return 'High'

instructor_df['instructor_tier'] = instructor_df['score'].apply(get_agg_tier)

# check how many batches they taught
counts = df.groupby('instructor_id').size().reset_index(name='batch_count')
instructor_df = pd.merge(instructor_df, counts, on='instructor_id')

instructor_df.head()